# Half-Heusler Seebeck: XGBoost Feature Importance + Symbolic Regression

**Step 1:** Train XGBoost on `half_heusler_SISSO_ready.csv`, get feature importance (built-in + SHAP)
**Step 2:** Run symbolic regression (PySR) to get an actual interpretable equation

If you already ran the previous notebook in this same Colab session, `df_sisso` / the CSV may still be in memory -- otherwise this notebook uploads the CSV fresh.

Run top to bottom.

## 1. Install dependencies

In [ ]:
!pip install xgboost shap pysr scikit-learn pandas -q

## 2. Load the data
Upload `half_heusler_SISSO_ready.csv` when prompted (skip this cell if `df_sisso` already exists from a previous cell in this session).

In [ ]:
import pandas as pd

try:
    df_sisso
    print("Using df_sisso already in memory.")
except NameError:
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df_sisso = pd.read_csv(fname)
    print(f"Loaded {fname}: {df_sisso.shape}")

## 3. Prepare X (features) and y (target)

In [ ]:
meta_cols = ["jid", "formula", "doping_type", "Seebeck"]
feature_cols = [c for c in df_sisso.columns if c not in meta_cols]

X = df_sisso[feature_cols].astype(float)
y = df_sisso["Seebeck"].astype(float)

print(f"Features used: {feature_cols}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

## 4. Train/test split + train XGBoost

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Test R2:  {r2_score(y_test, y_pred):.3f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.2f} uV/K")

## 5. Built-in XGBoost feature importance

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
print(importances.sort_values(ascending=False))

importances.plot(kind="barh", figsize=(7,5))
plt.xlabel("XGBoost feature importance (gain-based)")
plt.title("Half-Heusler Seebeck: feature importance")
plt.tight_layout()
plt.show()

## 6. SHAP values (more reliable than built-in importance -- shows direction + magnitude of effect per sample)

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer(X)

shap.summary_plot(shap_values, X, show=True)

mean_abs_shap = pd.Series(
    abs(shap_values.values).mean(axis=0), index=feature_cols
).sort_values(ascending=False)
print("Mean |SHAP value| per feature (overall effect size):")
print(mean_abs_shap)

## 7. Symbolic regression (PySR)
This searches for an actual closed-form equation relating the Seebeck coefficient to your input variables, instead of a black-box model. First run installs a Julia backend (one-time, ~2-3 min).

In [ ]:
import pysr
pysr.install()  # one-time Julia backend setup

In [ ]:
from pysr import PySRRegressor

sr_model = PySRRegressor(
    niterations=80,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square", "sqrt", "log", "exp"],
    populations=30,
    model_selection="best",   # balances accuracy vs. simplicity
    maxsize=20,                # cap expression complexity
    random_state=42,
)

sr_model.fit(X_train.values, y_train.values, variable_names=feature_cols)

print(sr_model)  # shows the Pareto front of equations (complexity vs. loss)

## 8. Evaluate the best symbolic equation on the test set

In [ ]:
best_eq = sr_model.get_best()
print("Best equation:")
print(best_eq["equation"])

y_pred_sr = sr_model.predict(X_test.values)
print(f"Symbolic model test R2:  {r2_score(y_test, y_pred_sr):.3f}")
print(f"Symbolic model test MAE: {mean_absolute_error(y_test, y_pred_sr):.2f} uV/K")

print("\nFull Pareto front (simpler <-> more accurate tradeoffs):")
print(sr_model.equations_[["complexity", "loss", "equation"]])